# DATT CLI launcher — Colab GPU
Run the cells in order on a fresh T4 runtime. Existing runtimes are preserved.
All runtime operations live in `scripts/datt.py`; the previous launcher is retained in
`DATT_Colab_GPU_legacy.ipynb`. See `docs/cli-operations.md`.

**Local changes not yet published must be copied into the checkout before use.** A clone of
GitHub main alone does not contain local edits. No automatic pull, commit, or push.
Configure `DATT_DATABASE_URL`, `DATT_STORAGE_BACKEND=supabase`, `SUPABASE_URL`,
`SUPABASE_SERVICE_ROLE_KEY`, and `DATT_STORAGE_BUCKET` (or `SUPABASE_STORAGE_BUCKET`)
in Colab Secrets with notebook access. Use the real project origin, not a placeholder.


In [ ]:
from pathlib import Path
import os, subprocess, sys
ROOT = Path('/content/DATT')
if not ROOT.exists():
    subprocess.run(['git', 'clone', '--branch', 'main', 'https://github.com/kraken2811/DATT.git', str(ROOT)], check=True, timeout=180)
os.chdir(ROOT)
assert (ROOT/'scripts/datt.py').is_file(), 'Checkout a revision containing the CLI or upload the source archive; see docs/cli-operations.md'
subprocess.run(['git', 'log', '-1', '--format=%h %s'], check=True, timeout=10)


In [ ]:
# Run only when provisioning dependencies; never while DATT is running.
subprocess.run([sys.executable, 'scripts/colab_install.py'], check=True, timeout=3700)


In [ ]:
# Secrets must be read in the notebook kernel, then inherited by CLI processes.
from scripts.colab_bootstrap import load_secrets
load_secrets()
os.environ['DATT_REQUIRE_PERSISTENCE'] = '1'
def datt(*args, timeout=180):
    subprocess.run([sys.executable, '-u', 'scripts/datt.py', *args], check=True, timeout=timeout)
datt('models', '--timeout', '600', timeout=650)
datt('bootstrap', '--mode', 'gpu', timeout=400)
datt('doctor', timeout=400)


In [ ]:
datt('start', '--mode', 'gpu')
from google.colab.output import serve_kernel_port_as_window
serve_kernel_port_as_window(8501)


In [ ]:
# Opt in: restarts only the CLI-owned backend, never the Colab runtime.
RUN_E2E = False
if RUN_E2E:
    datt('e2e', '--mode', 'gpu', '--timeout', '120', timeout=1800)


In [ ]:
# Optional operations (do not run automatically).
STOP_DATT = False
if STOP_DATT:
    datt('stop')
